

# LangGraph Revision Notes

## Core Idea

LangGraph is an orchestration framework for building stateful AI workflows and agents.

It is **not** the intelligence of the system. The LLM performs reasoning. LangGraph controls execution.

It manages:

* Workflow execution
* State management
* Conditional routing
* Tool orchestration
* Persistence
* Human in the loop
* Recovery from failures

---

# Why LangGraph Exists

A simple agent loop works for small demos:

```python
while True:
    response = llm.invoke(messages)

    if response.tool_calls:
        call_tool(...)
    else:
        break
```

Production systems need much more:

* Pause and resume
* Failure recovery
* Human approval
* Tool retries
* Persistent state
* Explicit routing
* Observability
* Parallel execution

LangGraph solves these production engineering problems.

---

# Graph Components

A LangGraph application consists of:

* State
* Nodes
* Edges

Everything revolves around these three concepts.

---

# Graph State

## Definition

State is the shared snapshot of everything the workflow currently knows.

Example:

```python
{
    "messages": [...],
    "intent": "...",
    "retrieved_docs": [...],
    "tool_results": {},
    "retry_count": 0
}
```

Every node:

* Reads from state
* Produces updates
* Returns only the fields it changed

The runtime merges those updates into the global state.

---

## Single Source of Truth

State should contain all information needed for execution.

Instead of scattering information across:

* Local variables
* Prompt strings
* Hidden class attributes
* Tool outputs

Everything important lives inside state.

Benefits:

* Easy debugging
* Easy replay
* Easy inspection
* Easier testing

---

## State vs Prompt

These are **not** the same.

State is application memory.

Prompt is only the subset of information sent to the LLM.

Not everything in state belongs in the prompt.

Examples:

* Internal IDs
* Permissions
* Retry counters
* Secrets

should usually remain outside the prompt.

---

# Nodes

A node performs one unit of work.

Examples:

* Intent classification
* Retrieval
* Tool execution
* Planning
* Validation
* Final answer generation

A node may contain:

* Python code
* LLM call
* Tool call
* Retriever
* Validator

---

## Good Node Boundaries

Create a separate node when a step has:

* Independent failure mode
* Independent retry policy
* External side effect
* Human approval boundary
* Separate model call

---

# Edges

Edges determine what executes next.

## Static Edge

```
Retrieve
    ↓
Answer
```

Always follows the same path.

---

## Conditional Edge

Execution depends on state.

Example:

```
Agent
   │
   ├── Tool
   ├── Clarification
   └── Finish
```

Conditional edges make routing explicit.

---

# Conditional Routing

Instead of hidden if else logic inside one giant function:

```
if tool:
    ...
elif clarify:
    ...
```

LangGraph exposes routing as part of the architecture.

Benefits:

* Easier debugging
* Easier testing
* Clear execution path
* Better observability

---

# Retrieval

Retrieval can be:

## Fixed

```
User
↓
Retrieve
↓
Answer
```

Useful when retrieval is always required.

---

## Agentic

```
User
↓
Agent

Retrieve?
Search?
SQL?
Answer directly?
```

The LLM decides whether retrieval is necessary.

---

## Hybrid

Simple requests:

```
Fast path
```

Complex requests:

```
Agentic retrieval
```

Often the best production tradeoff.

---

# Tool Calls

Typical loop:

```
Agent
↓
Tool
↓
Agent
↓
Tool
↓
Finish
```

This resembles the ReAct pattern.

Reason

↓

Act

↓

Observe

↓

Reason

---

# Persistence

Persistence


# Persistence

Persistence stores the execution state of the graph so that execution can continue later.

It enables:

* Crash recovery
* Long running workflows
* Human in the loop
* Replay
* State inspection
* Resumability

A graph is typically compiled with a checkpointer.

```
Graph
    ↓
Checkpoint
    ↓
Resume later
```

---

# Checkpoints

A checkpoint is a snapshot of the graph state at a particular point during execution.

It typically contains:

* Current state
* Next node(s)
* Thread ID
* Execution metadata

Checkpoints allow the graph to resume from the latest saved state instead of starting over.

---

# Thread ID

A thread represents one execution or conversation.

The thread ID is used to:

* Load previous state
* Resume execution
* Store checkpoints
* Continue conversations

Every checkpoint belongs to a thread.

---

# Supersteps

Execution happens in stages called supersteps.

Example:

```
START
  ↓
Node A
  ↓
Node B
  ↓
END
```

Independent nodes may execute within the same superstep.

```
        B
      ↗
A
      ↘
        C
```

B and C may execute in parallel.

---

# Checkpoint vs Long Term Memory

Checkpoint

* Stores workflow execution state
* Belongs to one thread
* Used for resuming execution

Long term memory

* Stores reusable information
* Shared across conversations
* Used for personalization

Checkpoint answers:

> "Where was execution paused?"

Long term memory answers:

> "What should the system remember about this user?"

---

# Interrupts

Interrupts allow execution to pause and resume later.

Typical use cases:

* Human approval
* User clarification
* External event
* Waiting for another system

Example:

```
Agent
↓
Refund Request
↓
Interrupt
↓
User Approval
↓
Resume
↓
Payment Tool
```

Interrupts make human in the loop a built in workflow concept.

---

# Checkpointing Does Not Solve External Side Effects

Checkpointing remembers graph execution.

It does **not** guarantee that external systems were updated safely.

Example:

```
Payment succeeds
↓

Server crashes

↓

Checkpoint not saved

↓

Graph retries payment
```

Without additional protection, the customer may be charged twice.

---

# Idempotency

Idempotency means:

> Executing the same operation multiple times produces the same final effect as executing it once.

Example:

```
Set status = Active
```

Run once

```
Active
```

Run ten times

```
Still Active
```

Still the same result.

---

## Non Idempotent Operation

```
balance = balance - 500
```

Run once

```
₹500 deducted
```

Run twice

```
₹1000 deducted
```

The effect changes every time.

---

# Idempotency Key

External systems often accept an idempotency key.

```
charge(
    amount=500,
    idempotency_key="order-123"
)
```

If the same request is received again:

```
Already processed
```

The payment service returns the previous result instead of charging again.

This prevents duplicate side effects during retries.

---

# Guardrails

Guardrails exist at multiple levels.

## Input Guardrails

* Authentication
* File validation
* Prompt injection detection
* Rate limiting

---

## Routing Guardrails

Validate every route before execution.

Never allow arbitrary node execution directly from model output.

---

## Tool Guardrails

* Typed arguments
* Permission checks
* Timeouts
* Allowlists
* Least privilege
* Confirmation before destructive actions

---

## Output Guardrails

* Citation verification
* Schema validation
* Safety filtering
* Groundedness checks

---

## Execution Guardrails

Prevent endless execution.

Example:

```
Maximum steps = 10
```

If exceeded:

```
Terminate safely
```

---

# Failure Modes

## Hallucination

Problem

The model invents information.

Solution

* Retrieval
* Tool verification
* Structured outputs
* Citation checking
* Deterministic validation

---

## Ambiguous Intent

Problem

```
Cancel it.
```

The request is incomplete.

Solution

Route to a clarification node instead of guessing.

---

## Tool Failure

Possible failures

* Timeout
* Authentication error
* Invalid arguments
* Rate limits
* Service unavailable

Solutions

* Retries
* Backoff
* Fallback route
* Error state
* Human review

---

## Infinite Loops

Example

```
Agent
↓

Search

↓

Agent

↓

Search

↓

...
```

Prevent using:

* Maximum step count
* Maximum token budget
* Timeout
* Duplicate action detection

---

# Major Tradeoffs

## While Loop vs Graph

While loop

* Simple
* Fast to prototype
* Hidden execution flow

Graph

* Explicit execution
* Easier debugging
* Persistence
* Human approval
* Better observability

---

## Latency vs Quality

More model calls

↓

Better reasoning

↓

Higher latency

Production systems often:

* Use small models for routing
* Use larger models only for difficult reasoning
* Skip unnecessary planning

---

## Cost vs Accuracy

Cheap model

* Intent classification
* Query rewriting
* Routing

Expensive model

* Planning
* Complex reasoning
* Final synthesis

---

## Autonomy vs Control

Low risk actions

* Read documents
* Search
* Draft responses

High risk actions

* Payments
* Sending emails
* Data deletion

High risk actions should use deterministic checks and often require human approval.

---

## Fine Grained vs Coarse Nodes

Fine grained

Advantages

* Better retries
* Better observability
* Better testing

Disadvantages

* More orchestration
* Higher latency
* More complexity

---

Coarse nodes

Advantages

* Simpler
* Faster

Disadvantages

* Harder debugging
* Larger retries
* Less visibility

---

# First Class Components

A first class component is something directly understood and supported by the framework.

Examples in LangGraph

* State
* Nodes
* Edges
* Conditional routing
* Checkpoints
* Interrupts

These are explicit concepts, not hidden implementation details.

---

# Why Graph Based Agents Matter

A graph based agent is not more powerful than a while loop.

A while loop can implement the same logic.

The advantage is architectural.

Graphs provide:

* Explicit execution flow
* Shared state
* Persistence
* Human in the loop
* Conditional routing
* Parallel execution
* Easier debugging
* Better observability
* Easier testing
* Safer retries

The graph transforms implicit control flow into explicit system architecture.

---

# Interview Takeaway

LangGraph is best viewed as an orchestration runtime rather than an LLM framework.

The LLM performs reasoning.

LangGraph manages:

* Execution
* State
* Routing
* Persistence
* Recovery
* Tool orchestration
* Human approval
* Observability

A production AI agent is not just a language model.

It is an execution system where the model is only one component.
